# Challenge - heat dynamics

In [ ]:
#    APM41012EP course notebook - Chapter 9 - M. Massot 2026-2027 - École polytechnique
#    ----------   
#    Challenge - heat dynamics
#    
#    Authors: L. Séries and M. Massot - (C) 2026

In [ ]:
from dataclasses import dataclass
import numpy as np
from scipy.integrate import solve_ivp
from scipy.sparse import diags
from scipy import sparse
from scipy.sparse.linalg import spsolve
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"

We consider the heat equation in 1D:

$$
\left\{
\begin{aligned}
& \partial_t u(x,t) - D \, \partial_{xx} u(x,t) = 0 \;\; \text{for} \; 0 < x < 1\\
& \partial_{x} u(0,t) = \partial_{x} u(1,t) = 0,\\
& u(x,t=t_0)  =  u_0(x),
\end{aligned}
\right.
$$

where the initial condition is chosen by the user, with homogeneous Neumann boundary conditions.

For the simulation, we take the initial time $t_0 =0$, two final times  $t_f=0.005$ and $t_f=0.015$, and $D=1$. We also take 1001 mesh points in space and 10001 points in time.

In [ ]:
#############################################
class heat_model:

    def __init__(self, d, xmin, xmax, nx) :
        self.d = d
        self.xmin = xmin
        self.xmax = xmax
        self.nx = nx
        self.dx = (xmax-xmin)/(nx-1)
        self.oneoverdxdx = 1/(self.dx*self.dx)
        
        doverdxdx = d/(self.dx**2)
        diag = np.repeat(-2, nx)
        sub_diag = np.repeat([1., 2.], [nx-2, 1])
        self.a = doverdxdx * diags([diag, sub_diag, np.flip(sub_diag)], [0, -1, 1])

    def fcn(self, t, y):
        return self.a.dot(y)
    
    def fcn_exact(self, t):
        xmin = self.xmin
        xmax = self.xmax
        d = self.d
        nx = self.nx
        dx = self.dx
        x = np.linspace(xmin, xmax, nx)
        y = (1/(2*np.sqrt(np.pi*t))) * np.exp(-(x*x)/(4.*t))
        return y

## Explicit Euler method

In [ ]:
@dataclass
class ode_result:
    t: np.ndarray 
    y: np.ndarray    
    
#############################################
def forward_euler(tini, tend, nt, yini, fcn):
    
    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yn = np.copy(yini)

    for it, tn  in enumerate(t[:-1]):
        yn = yn + dt*fcn(tn, yn)
        
    return ode_result(t, yn)

## Dynamics of the heat equation

We consider an initial condition with 2 modes

$u_0(x) = \cos\Big(2\pi x\Big) + 2 \cos\Big(6\pi x\Big)$


In [ ]:
xmin = 0.
xmax = 1.
nx = 1001

tini = 0.
tend = 0.005

dx = (xmax-xmin)/(nx-1)
x = np.linspace(xmin, xmax, nx)

d = 1.    
hm = heat_model(d=d, xmin=xmin, xmax=xmax, nx=nx)
fcn = hm.fcn

tini = 0.
tend = 0.005

k = 2*np.pi

a = 1
b = 2

yini = a*np.cos(k*x) + b*np.cos(3*k*x)

yexa1 = a*np.cos(k*x) * np.exp(-k**2 * tend)    + b*np.cos(3*k*x) * np.exp(-(3*k)**2 * tend)
yexa3 = a*np.cos(k*x) * np.exp(-k**2 * 3* tend) + b*np.cos(3*k*x) * np.exp(-(3*k)**2 * 3 * tend)

nt = 10001
dt = (tend-tini) / (nt-1)
print(f"dt = {dt}")

sol1 = forward_euler(tini, tend, nt, yini, fcn)
sol3 = forward_euler(tini, 3*tend, 3*(nt-1)+1, yini, fcn)

fig_sol = go.Figure()
fig_sol.add_trace(go.Scatter(x=x, y=yini, name="init. sol."))
fig_sol.add_trace(go.Scatter(x=x, y=yexa1, name="exact sol. (t=0.005)"))
fig_sol.add_trace(go.Scatter(x=x, y=yexa3, name="exact sol. (t=0.015)"))
fig_sol.add_trace(go.Scatter(x=x, y=sol1.y, name="num. sol. (t=0.005)"))
fig_sol.add_trace(go.Scatter(x=x, y=sol3.y, name="num. sol. (t=0.015)"))
fig_sol.update_layout(legend=dict(y=1.15, orientation="h"), height=450, title="Solution")
fig_sol.update_xaxes(title='x')
fig_sol.show()

fig_err = go.Figure()
fig_err.add_trace(go.Scatter(x=x, y=sol1.y-yexa1, name="err (t=0.005)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol3.y-yexa3, name="err (t=0.015)", showlegend=True))
fig_err.update_layout(legend=dict(y=1.20, orientation="h"), height=450, title="Error")
fig_err.update_yaxes(exponentformat='e')
fig_err.update_xaxes(title='x')
fig_err.show()

Interpret the dynamics quantitatively. What remains of the highest frequency at time $t=0.015$? Explain.

The goal of this challenge is to clearly understand the interplay between:
- the temporal error associated with a first-order scheme, which converges slowly,
- the spatial approximation error, which results in a modification of the relaxation rates of the various frequencies, due to the modification of the eigenvalues of the discrete Laplacian with respect to the eigenvalues of the Laplacian continuous in space,
- the multiscale relaxations of the amplitudes of the various frequencies present.

**Solution and errors for the lowest frequency and 10,000 time subdivisions for explicit Euler**

The lowest frequency can be isolated for this linear problem and its amplitude decays slowly according to an exponential whose argument is the associated eigenvalue $-\exp(−4 \pi^2 t)$ for the exact problem, approximated by an exponential relaxation associated with the approximate eigenvalue given by the spatial discretization. 
In the next cell, we display, for the lowest frequency, the solution, the error with respect to the exact solution and the error with respect to the quasi-exact solution obtained with a high-order implicit method (RadauIIA), which makes the associated time-resolution error negligible, leaving only the error associated with the spatial discretization so as to allow a precise analysis of the various contributions to the error.

In [ ]:
a = 1
b = 0

yini = a*np.cos(k*x) + b*np.cos(3*k*x)

yexa1 = a*np.cos(k*x) * np.exp(-k**2 * tend)    + b*np.cos(3*k*x) * np.exp(-(3*k)**2 * tend)
yexa3 = a*np.cos(k*x) * np.exp(-k**2 * 3*tend)  + b*np.cos(3*k*x) * np.exp(-(3*k)**2 * 3*tend)

sol_qexa = solve_ivp(fcn, (tini, 3*tend), yini, method='Radau', t_eval=[tend, 3*tend], rtol=1e-8, atol=1e-8)

nt = 10001
dt = (tend-tini) / (nt-1)
print(f"dt = {dt}")

sol1 = forward_euler(tini, tend, nt, yini, fcn)
sol3 = forward_euler(tini, 3*tend, 3*(nt-1)+1, yini, fcn)

fig_sol = go.Figure()
fig_sol.add_trace(go.Scatter(x=x, y=yini, name="init. sol."))
fig_sol.add_trace(go.Scatter(x=x, y=yexa1, name="exact sol. (t=0.005)"))
fig_sol.add_trace(go.Scatter(x=x, y=yexa3, name="exact sol. (t=0.015)"))
fig_sol.add_trace(go.Scatter(x=x, y=sol1.y, name="num. sol. (t=0.005)"))
fig_sol.add_trace(go.Scatter(x=x, y=sol3.y, name="num. sol. (t=0.015)"))
fig_sol.update_layout(legend=dict(y=1.15, orientation="h"), height=450, title="Solution")
fig_sol.update_xaxes(title='x')
fig_sol.show()

fig_err = go.Figure()
fig_err.add_trace(go.Scatter(x=x, y=sol1.y-yexa1, name="err (t=0.005)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol3.y-yexa3, name="err (t=0.015)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol1.y-sol_qexa.y[:,0], name="err (quasi-exact) (t=0.005)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol3.y-sol_qexa.y[:,1], name="err (quasi-exact) (t=0.015)", showlegend=True))
fig_err.update_layout(legend=dict(y=1.15, orientation="h"), height=450, title="Error")
fig_err.update_yaxes(title='(unum - uexa)', exponentformat='e')
fig_err.update_xaxes(title='x')
fig_err.show()

The error at the various times comes from a combination of the fact that the associated eigenvalue is approximated by that of the discrete Laplacian and of the use of an explicit Euler time scheme (the exact eigenvalue is larger in absolute value than the eigenvalue of the discrete Laplacian, but if the time step is not small enough, the temporal error compensates this dynamics). Having access to the quasi-exact solution makes it possible to converge in time and to keep only the spatial error stemming from the difference in eigenvalues.

**Solution and errors for the lowest frequency and 640,000 time subdivisions for explicit Euler**

We will be able to reach the same conclusion if we get closer to the quasi-exact solution by making the time scheme converge at fixed spatial discretization, dividing the time step by 128...

In [ ]:
a = 1
b = 0

yini = a*np.cos(k*x) + b*np.cos(3*k*x)

yexa1 = a*np.cos(k*x) * np.exp(-k**2 * tend)    + b*np.cos(3*k*x) * np.exp(-(3*k)**2 * tend)
yexa3 = a*np.cos(k*x) * np.exp(-k**2 * 3*tend)  + b*np.cos(3*k*x) * np.exp(-(3*k)**2 * 3*tend)

sol_qexa = solve_ivp(fcn, (tini, 3*tend), yini, method='Radau', t_eval=[tend, 3*tend], rtol=1e-8, atol=1e-8)

nt = 1280001
dt = (tend-tini) / (nt-1)
print(f"dt = {dt}")

sol1 = forward_euler(tini, tend, nt, yini, fcn)

sol3 = forward_euler(tini, 3*tend, 3*(nt-1)+1, yini, fcn)

fig_sol = go.Figure()
fig_sol.add_trace(go.Scatter(x=x, y=yini, name="init. sol."))
fig_sol.add_trace(go.Scatter(x=x, y=yexa1, name="exact sol. (t=0.005)"))
fig_sol.add_trace(go.Scatter(x=x, y=yexa3, name="exact sol. (t=0.015)"))
fig_sol.add_trace(go.Scatter(x=x, y=sol1.y, name="num. sol. (t=0.005)"))
fig_sol.add_trace(go.Scatter(x=x, y=sol3.y, name="num. sol. (t=0.015)"))
fig_sol.update_layout(legend=dict(y=1.15, orientation="h"), height=450, title="Solution")
fig_sol.update_xaxes(title='x')
fig_sol.show()

fig_err = go.Figure()
fig_err.add_trace(go.Scatter(x=x, y=sol1.y-yexa1, name="err (t=0.005)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol3.y-yexa3, name="err (t=0.015)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol1.y-sol_qexa.y[:,0], name="err (quasi-exact) (t=0.005)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol3.y-sol_qexa.y[:,1], name="err (quasi-exact) (t=0.015)", showlegend=True))
fig_err.update_layout(legend=dict(y=1.15, orientation="h"), height=450, title="Error")
fig_err.update_yaxes(title='(unum - uexa)', exponentformat='e')
fig_err.update_xaxes(title='x')
fig_err.show()

**Solution and errors for the highest frequency and 10,000 time subdivisions for explicit Euler**

The highest frequency can be isolated for this linear problem as well and its amplitude decays rapidly according to an exponential whose argument is the associated eigenvalue $-\exp(−36\pi^2 t)$ for the exact problem, approximated by an exponential relaxation associated with the approximate eigenvalue given by the spatial discretization.
In the next cell, we display, for the highest frequency, the solution, the error with respect to the exact solution and the error with respect to the quasi-exact solution obtained with a high-order implicit method (RadauIIA), which makes the associated time-resolution error negligible, leaving only the error associated with the spatial discretization so as to allow a precise analysis of the various contributions to the error.

In [ ]:
a = 0
b = 1

yini = a*np.cos(k*x) + b*np.cos(3*k*x)

yexa1 = a*np.cos(k*x) * np.exp(-k**2 * tend)    + b*np.cos(3*k*x) * np.exp(-(3*k)**2 * tend)
yexa3 = a*np.cos(k*x) * np.exp(-k**2 * 3*tend)  + b*np.cos(3*k*x) * np.exp(-(3*k)**2 * 3*tend)

sol_qexa = solve_ivp(fcn, (tini, 3*tend), yini, method='Radau', t_eval=[tend, 3*tend], rtol=1e-8, atol=1e-8)

nt = 10001
dt = (tend-tini) / (nt-1)
print(f"dt = {dt}")

sol1 = forward_euler(tini, tend, nt, yini, fcn)
sol3 = forward_euler(tini, 3*tend, 3*(nt-1)+1, yini, fcn)

fig_sol = go.Figure()
fig_sol.add_trace(go.Scatter(x=x, y=yini, name="init. sol."))
fig_sol.add_trace(go.Scatter(x=x, y=yexa1, name="exact sol. (t=0.005)"))
fig_sol.add_trace(go.Scatter(x=x, y=yexa3, name="exact sol. (t=0.015)"))
fig_sol.add_trace(go.Scatter(x=x, y=sol1.y, name="num. sol. (t=0.005)"))
fig_sol.add_trace(go.Scatter(x=x, y=sol3.y, name="num. sol. (t=0.015)"))
fig_sol.update_layout(legend=dict(y=1.15, orientation="h"), height=450, title="Solution")
fig_sol.update_xaxes(title='x')
fig_sol.show()

fig_err = go.Figure()
fig_err.add_trace(go.Scatter(x=x, y=sol1.y-yexa1, name="err (t=0.005)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol3.y-yexa3, name="err (t=0.015)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol1.y-sol_qexa.y[:,0], name="err (quasi-exact) (t=0.005)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol3.y-sol_qexa.y[:,1], name="err (quasi-exact) (t=0.015)", showlegend=True))
fig_err.update_layout(legend=dict(y=1.20, orientation="h"), height=450, title="Error")
fig_err.update_yaxes(exponentformat='e')
fig_err.update_xaxes(title='x')
fig_err.show()

The error at the various times comes from a combination of the fact that the associated eigenvalue is approximated by that of the discrete Laplacian and of the use of an explicit Euler time scheme (the exact eigenvalue is larger in absolute value than the eigenvalue of the discrete Laplacian, but if the time step is not small enough, the temporal error compensates this dynamics). Having access to the quasi-exact solution makes it possible to converge in time and to keep only the spatial error stemming from the difference in eigenvalues. In the case of this faster relaxation, the error decreases over time along with the solution.

**Solution and errors for the highest frequency and 2,560,000 time subdivisions for explicit Euler**

We will be able to reach the same conclusion if we get closer to the quasi-exact solution by making the time scheme converge at fixed spatial discretization, dividing the time step by 256...

In [ ]:
a = 0
b = 2

yini = a*np.cos(k*x) + b*np.cos(3*k*x)

yexa1 = a*np.cos(k*x) * np.exp(-k**2 * tend)    + b*np.cos(3*k*x) * np.exp(-(3*k)**2 * tend)
yexa3 = a*np.cos(k*x) * np.exp(-k**2 * 3*tend)  + b*np.cos(3*k*x) * np.exp(-(3*k)**2 * 3*tend)

sol_qexa = solve_ivp(fcn, (tini, 3*tend), yini, method='Radau', t_eval=[tend, 3*tend], rtol=1e-8, atol=1e-8)

nt = 2560001
dt = (tend-tini) / (nt-1)
print(f"dt = {dt}")

sol1 = forward_euler(tini, tend, nt, yini, fcn)
sol3 = forward_euler(tini, 3*tend, 3*(nt-1)+1, yini, fcn)

fig_sol = go.Figure()
fig_sol.add_trace(go.Scatter(x=x, y=yini, name="init. sol."))
fig_sol.add_trace(go.Scatter(x=x, y=yexa1, name="exact sol. (t=0.005)"))
fig_sol.add_trace(go.Scatter(x=x, y=yexa3, name="exact sol. (t=0.015)"))
fig_sol.add_trace(go.Scatter(x=x, y=sol1.y, name="num. sol. (t=0.005)"))
fig_sol.add_trace(go.Scatter(x=x, y=sol3.y, name="num. sol. (t=0.015)"))
fig_sol.update_layout(legend=dict(y=1.15, orientation="h"), height=450, title="Solution")
fig_sol.update_xaxes(title='x')
fig_sol.show()

fig_err = go.Figure()
fig_err.add_trace(go.Scatter(x=x, y=sol1.y-yexa1, name="err (t=0.005)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol3.y-yexa3, name="err (t=0.015)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol1.y-sol_qexa.y[:,0], name="err (quasi-exact) (t=0.005)", showlegend=True))
fig_err.add_trace(go.Scatter(x=x, y=sol3.y-sol_qexa.y[:,1], name="err (quasi-exact) (t=0.015)", showlegend=True))
fig_err.update_layout(legend=dict(y=1.20, orientation="h"), height=450, title="Error")
fig_err.update_yaxes(exponentformat='e')
fig_err.update_xaxes(title='x')
fig_err.show()

We then see that the solution converges to the quasi-exact solution, which allows us to exhibit the error associated with the spatial discretization. The error comes from the fact that the associated eigenvalue is approximated by that of the discrete Laplacian, except that in this case the much faster relaxation of the solution towards zero makes the error fade over time instead of growing. 

**Decay of the amplitude of the solution**

In [ ]:
b = 2
print("For the highest frequency ")
print(f" amplitude at t = 0.005: {b*np.exp(-(3*k)**2*0.005)}")
print(f" amplitude at t = 0.015: {b*np.exp(-(3*k)**2*0.015)}")

a = 1
print("\nFor the lowest frequency ")
print(f" amplitude at t = 0.005: {a*np.exp(-(k)**2*0.005)}")
print(f" amplitude at t = 0.015: {a*np.exp(-(k)**2*0.015)}")

We see that at time t=0.015 the exact amplitude has dropped below 1\%, whereas for the low frequency the amplitude is still 0.55.